In [0]:
-- DESCRIBE DETAILS OF TABLE (FORMAT, PARTITION COLUMNS, SIZE, LOCATION )
DESCRIBE DETAIL susep.bronze_ses_uf2;

In [0]:
-- SHOW PARTITIONS (COLUMNS AND VALUES)
SHOW PARTITIONS susep.bronze_ses_uf2;

In [0]:
-- GRUPOS DE RAMOS
SELECT
    gracodigo,
    granome
  FROM
    susep.bronze_ses_gruposramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_gruposramos
      )
    ORDER BY 1

In [0]:
 -- RAMOS
  SELECT
    coramo,
    noramo
  FROM
    susep.bronze_ses_ramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_ramos
      )

In [0]:
-- EXPLORATION RAMOS E GRUPOS
WITH cte_depara_ramos_grupo AS (
  SELECT DISTINCT
    ramos,
    gracodigo
  from
    susep.bronze_ses_uf2
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_uf2
      )
),
cte_ramos_grupo_agg as (
  SELECT
    ramos,
    gracodigo,
    count(*) over (partition by ramos) as total_grupo_ramos
  FROM
    cte_depara_ramos_grupo
),
cte_ramos AS (
  SELECT
    coramo,
    noramo
  FROM
    susep.bronze_ses_ramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_ramos
      )
),
cte_grupos_ramos AS (
  SELECT
    gracodigo,
    granome
  FROM
    susep.bronze_ses_gruposramos
  where
    ingestion_batch_id
      = (
        select
          max(ingestion_batch_id)
        from
          susep.bronze_ses_gruposramos
      )
)
SELECT
  cte_ramos.*,
  trim(regexp_replace(cte_ramos.noramo, '^[0-9]+\\s*-\\s*', '')) as nome_ramo,
  cte_grupos_ramos.*,
   trim(regexp_replace(cte_grupos_ramos.granome, '^[0-9]+\\s*-\\s*', '')) as nome_grupo_ramo
FROM
  cte_ramos_grupo_agg
    left join cte_ramos
      on cte_ramos_grupo_agg.ramos = cte_ramos.coramo
    left join cte_grupos_ramos
      on cte_ramos_grupo_agg.gracodigo = cte_grupos_ramos.gracodigo
ORDER BY nome_ramo


In [0]:
-- RAMOS NOT EXISTS IN TABLE UF
SELECT distinct
    coramo 
FROM susep.bronze_ses_seguros
where coramo not in (
  SELECT DISTINCT
    ramos
  from
    susep.bronze_ses_uf2
)

In [0]:
-- RAMOS NOT EXISTS IN TABLE RAMOS
SELECT 
    coramo,
    min(damesano) as min_damesano,
    max(damesano) as max_damesano 
FROM susep.bronze_ses_seguros
where
  ingestion_batch_id
    = (
      select
        max(ingestion_batch_id)
      from
        susep.bronze_ses_seguros
    )
  and coramo not in (
    SELECT DISTINCT
      coramo
    from
      susep.bronze_ses_ramos
    where
      ingestion_batch_id
        = (
          select
            max(ingestion_batch_id)
          from
            susep.bronze_ses_ramos
        )
  )
  group by coramo

In [0]:
-- RAMOS DISTINTCS
SELECT distinct
    coramo 
FROM susep.bronze_ses_seguros
where
  ingestion_batch_id
    = (
      select
        max(ingestion_batch_id)
      from
        susep.bronze_ses_seguros
    )

In [0]:
-- EMPRESAS

WITH base_limpa AS (
    SELECT DISTINCT
        coenti,
        TRIM(noenti) AS noenti,
        cogrupo,
        TRIM(nogrupo) AS nogrupo,
        UPPER(CONCAT(COALESCE(TRIM(nogrupo), ''), ' | ', COALESCE(TRIM(noenti), ''))) AS search_string
    FROM susep_bronze.ses_grupos_economicos
),

base_full AS
(
SELECT 
    *,
    CASE 
        -- 1. Grandes Conglomerados Bancários (Agrupando subsidiárias e marcas adquiridas)
        WHEN search_string LIKE '%BRADESCO%' OR search_string LIKE '%BCN%' OR search_string LIKE '%FINASA%' OR search_string LIKE '%BAMÉRCIO%' OR search_string LIKE '%BANERJ%' THEN 'BRADESCO SEGUROS'
        WHEN search_string LIKE '%ITAÚ%' OR search_string LIKE '%ITAU%' OR search_string LIKE '%UNIBANCO%' OR search_string LIKE '%BEMGE%' OR search_string LIKE '%BANCRED%' THEN 'ITAÚ UNIBANCO'
        WHEN search_string LIKE '%SANTANDER%' OR search_string LIKE '%REAL%' OR search_string LIKE '%HSBC%' OR search_string LIKE '%BAMERINDUS%' OR search_string LIKE '%AMÉRICA DO SUL%' OR search_string LIKE '%BBV%' THEN 'SANTANDER'
        WHEN search_string LIKE '%CAIXA%' THEN 'CAIXA SEGURIDADE'
        WHEN search_string LIKE '%SAFRA%' THEN 'SAFRA'
        WHEN search_string LIKE '%BANCO PACTUAL%' OR search_string LIKE '%PACTUAL%' THEN 'BTG PACTUAL'
        
        -- 2. Grandes Seguradoras Nacionais e Internacionais
        WHEN search_string LIKE '%PORTO SEGURO%' THEN 'PORTO SEGURO'
        WHEN search_string LIKE '%SUL AMERICA%' OR search_string LIKE '%SULAMÉRICA%' OR search_string LIKE '%SULINA%' THEN 'SULAMÉRICA'
        WHEN search_string LIKE '%MAPFRE%' OR search_string LIKE '%BBMAPFRE%' THEN 'MAPFRE'
        WHEN search_string LIKE '%ALLIANZ%' OR search_string LIKE '%AGF%' THEN 'ALLIANZ'
        WHEN search_string LIKE '%ZURICH%' OR search_string LIKE '%MINAS-BRASIL%' THEN 'ZURICH'
        WHEN search_string LIKE '%CHUBB%' OR search_string LIKE '%ACE%' OR search_string LIKE '%NOVA YORK%' THEN 'CHUBB'
        WHEN search_string LIKE '%LIBERTY%' OR search_string LIKE '%INDIANA%' OR search_string LIKE '%PAULISTA%' THEN 'LIBERTY'
        WHEN search_string LIKE '%TOKIO MARINE%' THEN 'TOKIO MARINE'
        
        -- 3. Grupos de Saúde / Operadoras Grandes
        WHEN search_string LIKE '%AMIL%' OR search_string LIKE '%BLUE LIFE%' OR search_string LIKE '%UNITED%' THEN 'AMIL'
        WHEN search_string LIKE '%UNIMED%' THEN 'UNIMED'
        WHEN search_string LIKE '%NOTRE DAME%' OR search_string LIKE '%GOLDEN CROSS INTERMÉDICA%' THEN 'NOTRE DAME INTERMÉDICA'
        
        -- 4. Grandes Especializadas e Tradicionais Consolidadas
        WHEN search_string LIKE '%HDI%' OR search_string LIKE '%HANNOVER%' OR search_string LIKE '%TALANX%' THEN 'HDI SEGUROS'
        WHEN search_string LIKE '%POTTENCIAL%' THEN 'POTTENCIAL'
        WHEN search_string LIKE '%EULER HERMES%' THEN 'EULER HERMES'
        WHEN search_string LIKE '%YASUDA%' OR search_string LIKE '%MARÍTIMA%' THEN 'SOMPO SEGUROS'
        WHEN search_string LIKE '%GENERALI%' OR search_string LIKE '%GENERALLI%' THEN 'GENERALI'
        WHEN search_string LIKE '%METROPOLITAN%' OR search_string LIKE '%METLIFE%' THEN 'METLIFE'
        WHEN search_string LIKE '%CARDIF%' OR search_string LIKE '%LUIZASEG%' THEN 'BNP PARIBAS CARDIF'
        WHEN search_string LIKE '%ICATU%' THEN 'ICATU'
        WHEN search_string LIKE '%SABEMI%' THEN 'SABEMI'
        WHEN search_string LIKE '%GBOEX%' OR search_string LIKE '%CONFIANCA%' THEN 'GBOEX-CONFIANÇA'

        -- O Funil Estrito: Tudo o que for pulverizado, independente antigo ou genérico vira 'OUTROS'
        ELSE 'OUTROS'
    END AS nome_empresa_consolidada

FROM base_limpa)

SELECT DISTINCT 
    *
FROM base_full
;
